# New section

In [193]:
!pip install -U langchain-openai
!pip install python-dotenv
!pip install python-dotenv langchain openai faiss-cpu
!pip install langchain openai faiss-cpu
!pip install --upgrade langchain
!pip install langchain_community
!pip install langchain_openai

ERROR: Operation cancelled by user


# New section

In [194]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain.llms import OpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import CSVLoader
from langchain.chat_models import ChatOpenAI
from google.colab import userdata

In [195]:
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

In [196]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [197]:
os.environ.pop('OPENAI_API_KEY', None)


openai_api_key = 'sk-ebdM8qcZ6KODTqhglzCiTM1hRWg9JgIHNWVhpsqbg4T3BlbkFJErmk6gObezeGCLdHfZ8aMbJsxw6qx1tukdniAtY2cA'

print(f"OpenAI API Key: {openai_api_key}")

if not openai_api_key or not openai_api_key.startswith('sk-'):
    raise ValueError("OpenAI API key is missing or invalid")

OpenAI API Key: sk-ebdM8qcZ6KODTqhglzCiTM1hRWg9JgIHNWVhpsqbg4T3BlbkFJErmk6gObezeGCLdHfZ8aMbJsxw6qx1tukdniAtY2cA


In [198]:
#data = pd.read_excel(r'C:\Users\cheww\Documents\Y3S1\DIP Project\modsoptimizer.xlsx')

In [227]:
file_path = (r"/content/modsoptimizerv3.csv")
#df = pd.read_csv(file_path)
#loader = [Document(page_content=row.to_string()) for _, row in df.iterrows()]
loader = CSVLoader(file_path=file_path, csv_args={"delimiter": ",",})
data = loader.load()
print(len(data))

2309


In [228]:
#data_text = ""
#for index, row in data.iterrows():
#    row_text = " | ".join([f"{col}: {row[col]}" for col in data.columns])
#    data_text += row_text + "\n"

In [229]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, add_start_index=True)
texts = text_splitter.split_documents(data)

In [230]:
texts_for_faiss = [doc.page_content for doc in texts]
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=openai_api_key)

In [231]:
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
vector_store.add_documents(documents=texts)

['e3bbc7fd-e54f-4816-a92e-cf9d09af668a',
 '06519175-a6f0-4f2e-9111-f19e66915820',
 '16b33030-0376-455c-95dd-1b444c81a19d',
 '23057144-6358-45a6-b3af-9f597d0d8cca',
 'b2491523-220f-404c-a3a5-0a2639c6f702',
 'a5ef6f93-83ee-4092-bf0d-86357c728ddb',
 '0125d62f-a3ab-4cc9-a227-bee092a4603e',
 '599b3e4f-dd7f-4fda-a9d0-575909f5c7eb',
 '2e0c32aa-fb93-42bd-9ffc-a31bbb064357',
 '64650d8b-bbe4-49b3-b01a-752d773d1a85',
 '76252740-e77c-4fce-b614-85c8371ed54a',
 'fea0212e-4bec-4544-8e36-dd6af08d013b',
 '9766b9fd-3ec1-4849-ac05-73304bb90fe2',
 'e06d42c5-7664-40bd-922d-fb9adb1c0ed8',
 '2c449a90-857b-46bc-8190-4f7c5ba685ee',
 '0fa1c892-981b-4bc1-8ffe-231baa69c0e6',
 'ba12855f-fc49-4f5b-9ad4-30bc5f396f7a',
 '8d5b6417-de05-4d3a-b98b-6c6a692b3208',
 '3dd27cfc-860c-4329-a601-e1d218dafeb9',
 '0cb327d2-ffd7-4d8e-a612-466f52ecd943',
 'cb90d8b2-634b-4db0-b014-d14eae9f2ed0',
 'bcc2eb43-22ed-4f82-a5e3-a0ea93a94a51',
 '6e648e36-7dbf-41d4-8989-b14abd3c2dd8',
 '5aa6ff9d-a0b9-4ad9-802c-c7f54262297f',
 '53054135-81b1-

In [232]:
llm = ChatOpenAI(
    openai_api_key=openai_api_key,
    model="gpt-3.5-turbo",
    temperature=0,  # Adjust for creative vs factual answer
    max_tokens = 1400
)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# New section

In [248]:
question = "what is the best eee module"

In [249]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_keywords(question):
    doc = nlp(question)
    keywords = []
    unwanted_words = {"course", "code", "name", "description", "list"}
    # Extract nouns and proper nouns as keywords, excluding unwanted words
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "ADJ"] and token.text.lower() not in unwanted_words:
            keywords.append(token.text)

    # Remove duplicates by converting the list to a set
    unique_keywords = set(keywords)
    return " ".join(unique_keywords)
keywords = extract_keywords(question)

print(keywords)


def is_course_related(question):
    course_keywords = ["introduce","prereq", "prerequisite", "AU", "details", "course", "recommend", "subject", "class", "module", "NTU", "major", "elective", "learn","mod", "BDE", ""]
    doc = nlp(question.lower())

    # Check if any of the course-related keywords are present in the question
    for token in doc:
        if token.text in course_keywords:
            return True
    return False

best module eee


In [250]:
def setup_llm_chain():
    prompt_template = PromptTemplate(
        input_variables=["chat_history", "context"],
        template="""
        You are an expert assistant. Here's the conversation so far:
        {chat_history}
        Now, use the following context to answer the question:
        {context}
        Provide a helpful and accurate answer.

        Here some more information you need to consider regarding column provided in the data:

Core: module that must be taken by the major
BDE is Broadening deepening electives. These are the module that is available to students outside of their core to be taken.

A student can not take a BDE from a module that are ran from their department. For instance, if you are a EEE student you can not take EE3101 as a BDE
Finally, if you are asked about details regarding a certain module please provide the course code, description, academic units, course title and prerequisite and dont include level


If you are given questions that is related to subjective judgements, please provide a disclaimer that you dont have the exact data to backup your statement. such as when you are asked about which is the best mod


        """
    )

    llm_chain = LLMChain(
        prompt=prompt_template,
        llm=llm,
        memory=memory
    )
    return llm_chain

In [251]:
def chat_with_llm_chain(question):
    # Classify if the question is related to course
    keywords = extract_keywords(question)
    retriever = vector_store.as_retriever(
    search_type="similarity", search_kwargs={'k': 50}
    )
    doc = retriever.invoke(keywords)
    if is_course_related(question):
        # Create an instruction for course recommendation scenario
        context = f"Instruction: You are a helpful assistant designed to help NTU students find courses. Provide accurate information about available courses at Nanyang Technological University.\n\nDocuments: {doc}\n\nQuestion: {question}"

        # Set up QA chain for course-related question
        llm_chain = setup_llm_chain()
        response = llm_chain.invoke({"context": context})
        return response
    else:
        # For non-course-related questions, use GPT to answer directly
        context = f"Instruction: You are a general-purpose assistant. Answer the following question accurately and helpfully.\n\nQuestion: {question}"
        llm_chain = setup_llm_chain()
        response = llm_chain.invoke({"context": context})
        return response

In [252]:
response = chat_with_llm_chain(question)
print(response['text'])

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 19247 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}